# Crecimiento poblacional

## Teoria

Explicacion ecuacion

$\frac{dP}{dt} = rP( 1 - \frac{P}{K})$

## Código

### Bibliotecas

In [ ]:
import sympy as sp
import numpy as np
import plotly.graph_objects as go
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import ipywidgets as widgets
from IPython.display import display, Math

### Funciones generales

In [ ]:
def analizar_ecuacion_diferencial(ecuacion, y):
    print("Ecuación diferencial planteada:")
    display(ecuacion)

    clasificacion = sp.classify_ode(ecuacion, y)
    es_lineal = any('linear' in c for c in clasificacion)
    es_ordinaria = any('ordinary' in c for c in clasificacion)
    orden = sp.ode_order(ecuacion, y)

    print("Clasificaciones de la ecuación diferencial:")
    print(f"Es lineal: {es_lineal}")
    print(f"Es ordinaria: {es_ordinaria}")
    print(f"Orden: {orden}")

    print("Posibles métodos de solución:")
    for metodo in clasificacion:
        print(f"- {metodo}")

In [ ]:
def solucion_general(ecuacion, r_val, K_val):
    sol_general = sp.dsolve(ecuacion, P)
    if isinstance(sol_general, list):
        sol_general = sol_general[0]

    sol_general = sol_general.subs({
        r: r_val,
        K: K_val
    })
    sol_general = sol_general.replace(
        lambda x: x.is_Float,
        lambda x: round(float(x), 2)
    )

    return sol_general

In [ ]:
def solucion_particular(solucion_general, y0):
    expresion = solucion_general.rhs
    C = next(iter(expresion.free_symbols - {t}))

    condicion_inicial = sp.Eq(expresion.subs(t, 0), y0)
    soluciones_C = sp.solve(condicion_inicial, C, check=False)

    if not soluciones_C:
        condicion_inicial = sp.powdenest(condicion_inicial, force=True)
        soluciones_C = sp.solve(condicion_inicial, C, check=False)

    if not soluciones_C:
        raise ValueError("No se pudo determinar la constante de integración.")

    solucion_sin_raices = sp.simplify(expresion.subs(C, soluciones_C[0])).replace(
        lambda e: (
            isinstance(e, sp.Pow)
            and e.exp.is_Rational
            and e.exp.q % 2 == 1
        ),
        lambda e: sp.real_root(e.base, e.exp.q) ** e.exp.p
    ).replace(
        lambda x: x.is_Float,
        lambda x: round(float(x), 2)
    )
    solucion_particular = sp.Eq(P, solucion_sin_raices)

    return solucion_particular

In [ ]:
def graficar_campo_direccional_ecuacion(ecuacion, x_min=0, x_max=15, y_min=-15, y_max=15):
    n_puntos = 30
    
    dydx = sp.lambdify((t, P), ecuacion.rhs, modules='numpy')
    
    x_vals = np.linspace(x_min, x_max, n_puntos)
    y_vals = np.linspace(y_min, y_max, n_puntos)
    X, Y = np.meshgrid(x_vals, y_vals)

    U = np.ones_like(X)
    V = dydx(X, Y)
    V = np.where(np.isfinite(V), V, np.nan)

    magnitud = np.sqrt(U**2 + V**2)
    magnitud = np.where(magnitud == 0, 1, magnitud)

    U_norm = U / magnitud
    V_norm = V / magnitud

    angulo = np.arctan2(V_norm, U_norm)
    angulo = (angulo + 2 * np.pi) % (2 * np.pi)

    normalizacion_absoluta = mcolors.Normalize(vmin=0, vmax=2*np.pi)

    fig, ax = plt.subplots(figsize=(9, 5))

    ax.quiver(X, Y, U_norm, V_norm, angulo, cmap='RdBu', norm=normalizacion_absoluta, pivot='mid', scale=50)
    ax.set_title("Campo direccional de la ecuación diferencial")
    ax.set_xlabel("t")
    ax.set_ylabel("P(t)")
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    plt.close(fig)

    return fig

In [ ]:
def graficar_solucion_particular(solucion_particular, x_min=0, x_max=15):
    n_puntos = 200
    
    f = sp.lambdify(t, solucion_particular.rhs, modules='numpy')

    t_vals = np.linspace(x_min, x_max, n_puntos)
    P_vals = f(t_vals)

    P_vals = np.where(np.isfinite(P_vals), P_vals, np.nan)
    #P_vals = np.where(np.isreal(P_vals), P_vals, np.nan)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(t_vals, P_vals, color='blue', label='Solución particular')
    ax.set_title("Solución particular de la ecuación diferencial")
    ax.set_xlabel("t")
    ax.set_ylabel("P(t)")
    ax.set_xlim(x_min, x_max)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)

    plt.close(fig)
    
    return fig

### Interfaz

In [ ]:
slider_r = widgets.FloatSlider(
    value=0.01,
    min=0.01,
    max=1.0,
    step=0.01,
    description='r:',
    continuous_update=False
)

slider_K = widgets.IntSlider(
    value=250,
    min=100,
    max=4000,
    step=100,
    description='K:',
    continuous_update=False
)

slider_y0 = widgets.IntSlider(
    value=10,
    min=1,
    max=5000,
    step=10,
    description='P(0):',
    continuous_update=False
)

slider_t_max = widgets.IntSlider(
    value=100,
    min=1,
    max=2000,
    step=1,
    description='t max:',
    continuous_update=False
)

slider_p_max = widgets.IntSlider(
    value=500,
    min=1,
    max=500,
    step=10,
    description='P(t) max:',
    continuous_update=False
)

In [ ]:
salida_general = widgets.Output()
salida_campo = widgets.Output()
salida_particular = widgets.Output()
salida_grafica = widgets.Output()

In [ ]:
fila_1 = widgets.HBox([
    slider_r,
    slider_K,
    slider_y0
])

fila_2 = widgets.HBox([
    slider_t_max,
    slider_p_max
])

columna_izquierda = widgets.VBox(
    [
        salida_general,
        salida_campo
    ],
    layout=widgets.Layout(
        width='50%',
        padding='10px'
    )
)

columna_derecha = widgets.VBox(
    [
        salida_particular,
        salida_grafica
    ],
    layout=widgets.Layout(
        width='50%',
        padding='10px'
    )
)

resultado = widgets.HBox(
    [
        columna_izquierda,
        columna_derecha
    ],
    layout=widgets.Layout(
        width='100%'
    )
)

In [ ]:
def mostrar_solucion(r_val, K_val, t_max_val, p_max_val, y0_val):
    ecuacion_con_parametros = ecuacion.subs({
        r: r_val,
        K: K_val
    })

    solucion_gnr = solucion_general(ecuacion_con_parametros, r_val, K_val)

    print("Solución general de la ecuación diferencial:")
    print()
    with salida_general:
        salida_general.clear_output()
        display(Math(
        sp.latex(solucion_gnr)
    ))

    print("Campo direccional de la ecuación diferencial:")
    print()

    grafica_campo = graficar_campo_direccional_ecuacion(ecuacion_con_parametros, x_min=0, x_max=t_max_val, y_min=0, y_max=p_max_val)
    with salida_campo:
        salida_campo.clear_output()
        display(grafica_campo)

    solucion_part = solucion_particular(solucion_gnr, y0_val)
    print("Solución particular de la ecuación diferencial:")
    print()
    with salida_particular:
        salida_particular.clear_output()
        display(Math(
        sp.latex(solucion_part)
    ))

    print("Gráfica de la solución particular:")
    grafica_solucion = graficar_solucion_particular(
        solucion_part,
        x_min=0,
        x_max=t_max_val
    )
    with salida_grafica:
        salida_grafica.clear_output()
        display(grafica_solucion)

### Definir ecuacion

In [ ]:
t = sp.Symbol('t')
P = sp.Function('P')(t)
r = sp.Symbol('r')
K = sp.Symbol('K')
derivada = sp.Derivative(P, t)

In [ ]:
ecuacion = sp.Eq(derivada, r * P * (1 - P / K))

## Analizar ecuacion

In [ ]:
analizar_ecuacion_diferencial(ecuacion, P)

## Laboratorio interactivo

In [ ]:
salida = widgets.interactive_output(
    mostrar_solucion,
    {
        'r_val': slider_r,
        'K_val': slider_K,
        't_max_val': slider_t_max,
        'p_max_val': slider_p_max,
        'y0_val': slider_y0
    }
)

display(
    widgets.VBox([
        fila_1,
        fila_2,
        resultado
    ])
)